# The Service Runtime

`EpiLensRuntime` assembles retrieval and workflows behind `explore`, `classify`, and `precision_mine`. The API, CLI, and UI use this service layer. This notebook shows those methods on an offline sample; notebook 09 shows how a workspace supplies runtime settings.


In [ ]:
import sys
from pathlib import Path

if (Path.cwd() / "notebooks" / "epilens_nb.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "notebooks"))
import epilens_nb as nb
WORK_DIR = nb.bootstrap()


## Prepare the corpus

The runtime expects an index and paper metadata. The helper builds both from two short papers with a deterministic local embedder.


In [ ]:
from epilens.services import EpiLensRuntime, RuntimeConfig

papers = nb.sample_papers()
strategy_name = "runtime-demo"
index_dir = WORK_DIR / "index"
metadata_file = WORK_DIR / "academic_db.json"
_, retriever = nb.build_index(index_dir, papers, nb.TinyKeywordEmbedder(nb.SHARING_VOCABULARY))
academic_db = nb.build_academic_db(papers, strategy_name, backup_file=metadata_file)


## Configure once

`RuntimeConfig` normally comes from app settings. With no Qdrant or MongoDB URL, the runtime uses local file storage. We pass the notebook retriever below because its tiny embedder is a teaching stand-in, not a registered model backend.


In [ ]:
config = RuntimeConfig(
    strategy_name=strategy_name,
    index_dir=index_dir,
    metadata_backup=metadata_file,
    workflow_top_k=4,
)
runtime = EpiLensRuntime(config)
health = runtime.health()
print("Vector backend:", health.checks["vector_backend"])
print("Metadata papers:", runtime.build_db().list_docs(strategy_name))


## Explore, classify, and mine

`explore` retrieves passages. `classify` returns labels from a fixed vocabulary; `precision_mine` returns a list of extracted items. The latter two use scripted JSON here, so their outputs demonstrate the interfaces rather than model quality.


In [ ]:
found = runtime.explore("Can the patient data be shared?", top_k=2, retriever=retriever)
for chunk in found.retrieved_chunks:
    print(f"{chunk.paper_id} | {chunk.section_title}: {chunk.text[:120]}")


In [ ]:
classification = runtime.classify(
    "paper_closed_data",
    classifier_kind="data_accessibility",
    retriever=retriever,
    academic_db=academic_db,
    generator=nb.FixedJSONGenerator({
        "classification": ["E"],
        "primary_label": "E",
        "reasoning": "The paper explicitly restricts sharing the patient data.",
    }),
)
print("Reported access:", [label.value for label in classification.decision.result.classification])

extraction = runtime.precision_mine(
    "paper_open_data",
    miner_kind="find_data_sources",
    retriever=retriever,
    academic_db=academic_db,
    generator=nb.FixedJSONGenerator({
        "description": "One named registry appears in the methods.",
        "items": [{
            "name": "National Hospital Registry", "url": None,
            "explanation": "Named as the patient-record source.",
            "raw_text": "The analysis used patient records from the National Hospital Registry.",
        }],
    }),
)
print("Data sources:", [item.name for item in extraction.result.items])


The runtime resolves workflow names such as `data_accessibility` and `find_data_sources` through the same registry used by other entry points. With a real embedder and generator configured, callers can let it build those components. Check retrieved evidence before relying on labels or extracted items; the fixed JSON in this notebook ignores that evidence.
